### File Handling in Python : Your Complete Guide to Reading, Writing, and Managing Files

Imagine your Python program is a goldfish and it has a **terrible memory**. The moment you stop running it, everything it calculated, every user it met, every score it tracked is **gone**.

**Files are the solution.** They let your programs permanently store data on your computer's hard disk, so information survives even after the program closes.

**Real-world examples of file handling in action:**
- **Netflix** stores your watch history, ratings, and preferences in files/databases on their servers
- **Google Docs** auto-saves your documents to files every few seconds so you never lose work
- **Spotify** reads your playlist and song metadata from files when you open the app
- **WhatsApp** stores your entire chat history in encrypted files on your device
- **Amazon** processes millions of order records stored in CSV/JSON files every day

**What we will learn:**
1. Opening, reading, writing, and closing files
2. All file modes (`r`, `w`, `a`, `x`, `b`, `+`) : what they mean and when to use them
3. Navigating inside a file using `seek()` and `tell()`
4. Managing directories and file paths with the `os` module
5. Modern path handling with `pathlib`
6. Working with CSV and JSON files : the most common file types in Data Science
7. Handling file errors gracefully
8. Copying and moving files with `shutil`


### **Creating Sample Files for This Notebook**

Before we dive in, let us create a few sample files that we will use throughout the examples. Think of this as setting up the lab before the experiment!

In [1]:
import os

# Create a folder to keep our practice files tidy
os.makedirs("file_handling_demo", exist_ok=True)

# Sample text file
with open("file_handling_demo/learn.txt", "w", encoding="utf-8") as f:
    f.write("Day 1: Started learning Python file handling!\n")
    f.write("Day 2: Understood the 'with' statement. Life changing!\n")
    f.write("Day 3: Read and wrote my first CSV file like a data scientist!\n")

print("Sample files created successfully!")
print("We are ready to go.")


Sample files created successfully!
We are ready to go.


### **1. Opening a File** 

Before you can read or write a file, you must **open** it first. Think of it like opening a book for reading.

Python's built-in `open()` function does this job.

```python
file_object = open(filename, mode, encoding)
```

**Parameters:**
| Parameter | Description | Default |
|-----------|-------------|---------|
| `filename` | Path to the file (absolute or relative) | Required |
| `mode` | How you want to open it (`'r'`, `'w'`, `'a'`, etc.) | `'r'` (read) |
| `encoding` | Character encoding to use | Platform-dependent |

The function returns a **file object** (also called a file handle)

> **Pro tip:** Always specify `encoding='utf-8'` explicitly. The default encoding varies by OS (Windows uses `cp1252`, Linux/Mac use `utf-8`). Being explicit makes your code portable!

In [4]:
# Opening a file
f = open("file_handling_demo/learn.txt", "r", encoding="utf-8")

print("File opened successfully!")
print("Type of file object:", type(f))
f.close()  # Always close what you open

File opened successfully!
Type of file object: <class '_io.TextIOWrapper'>


### **2. File Modes : Telling Python What You Want to Do**

The **mode** is the most important argument after the filename. It tells Python *what you plan to do* with the file.

Think of it like booking a seat at a restaurant:
- `'r'` : You're just going to read the menu (no ordering)
- `'w'` : You're going to create a brand new menu (existing one gets thrown away!)
- `'a'` : You want to add new items to the existing menu without changing the rest
- `'x'` : You want to create a new menu, but only if one doesn't already exist

### Mode Quick Reference

| Mode | Full Name | Creates file? | Overwrites? | Can Read? | Can Write? |
|------|-----------|:---:|:---:|:---:|:---:|
| `'r'` | Read | ❌ | ❌ | ✅ | ❌ |
| `'w'` | Write | ✅ | ✅ | ❌ | ✅ |
| `'a'` | Append | ✅ | ❌ | ❌ | ✅ |
| `'x'` | Exclusive create | ✅ (fails if exists) | ❌ | ❌ | ✅ |
| `'r+'` | Read + Write | ❌ | ❌ | ✅ | ✅ |
| `'w+'` | Write + Read | ✅ | ✅ | ✅ | ✅ |
| `'a+'` | Append + Read | ✅ | ❌ | ✅ | ✅ |

### Text vs Binary

Add `'b'` to any mode to open in **binary mode** (for images, videos, audio, etc.):
- `'rb'` : Read a binary file (like reading a photo)
- `'wb'` : Write a binary file (like saving a downloaded image)

In [12]:
# 'r' mode : Read only (file must already exist)
f = open("file_handling_demo/learn.txt", "r", encoding="utf-8")
print("'r'readable:", f.readable(), "| writable:", f.writable())
f.close()

# 'w' mode : Write only (creates or WIPES the file!)
f = open("file_handling_demo/temp_w.txt", "w", encoding="utf-8")
print("'w'readable:", f.readable(), "| writable:", f.writable())
f.close()

# 'a' mode : Append only (creates or adds to file)
f = open("file_handling_demo/temp_a.txt", "a", encoding="utf-8")
print("'a'readable:", f.readable(), "| writable:", f.writable())
f.close()

# 'r+' mode : Read AND write (file must exist; cursor starts at beginning)
f = open("file_handling_demo/learn.txt", "r+", encoding="utf-8")
print("'r+'readable:", f.readable(), "| writable:", f.writable())
f.close()

# 'x' mode : Exclusive creation (fails if file already exists)
try:
    f = open("file_handling_demo/brand_new.txt", "x", encoding="utf-8")
    f.write("I am a brand new file!\n")
    f.close()
    print("'x' mode, file created successfully!")
except FileExistsError:
    print("'x' mode, file already exists, creation failed (as expected)!")

'r'readable: True | writable: False
'w'readable: False | writable: True
'a'readable: False | writable: True
'r+'readable: True | writable: True
'x' mode, file already exists, creation failed (as expected)!


### **3. Closing a File**

When you open a file, the operating system allocates **resources** to manage it. If you forget to close it:
- Memory is wasted
- Other programs may not be able to access the file
- Data you wrote might not be saved (buffering issues)

Think of it like borrowing a book from a library which you must return.

#### Three ways to close files:

**Method 1 : `close()` (risky):**
If an error occurs before `f.close()`, the file stays open forever.

**Method 2 : `try/finally` (safer):**
Guarantees closure even if an error occurs.

**Method 3 : `with` statement (best!):**
Automatically closes the file when the block ends. This is the industry standard.


In [15]:
# Method 1: Manual close
f = open("file_handling_demo/learn.txt", "r", encoding="utf-8")
content = f.read()
f.close()
print("File closed:", f.closed)

File closed: True


In [17]:
# Method 2: try/finally
try:
    f = open("file_handling_demo/learn.txt", "r", encoding="utf-8")
    content = f.read()
finally:
    f.close()
print("File closed:", f.closed)

File closed: True


In [19]:
# Method 3: 'with' statement
with open("file_handling_demo/learn.txt", "r", encoding="utf-8") as f:
    content = f.read()
# File is automatically closed here, no matter what!

print("File closed:", f.closed)
print("\nThe 'with' statement is the professional way to handle files. So we will use it always.")

File closed: True

The 'with' statement is the professional way to handle files. So we will use it always.


### **4. Writing to a File**

Python gives you two main methods to write data to a file:

##### `write(string)` 
Writes a single string to the file. Returns the number of characters written.
- Does **NOT** add a newline automatically, you must add `\n` yourself
- Think of it like typing on a typewriter: you control every character

##### `writelines(list_of_strings)`
Writes a list of strings to the file all at once. More efficient when you have many lines.
- Also does **NOT** add newlines automatically, have to include `\n` in each string
- Think of it like stamping a whole stack of pages at once


In [23]:
# write() : writing strings one at a time
with open("file_handling_demo/shopping_list.txt", "w", encoding="utf-8") as f:
    chars = f.write("My Shopping List:\n")
    
    f.write("1. Milk\n")
    f.write("2. Eggs\n")
    f.write("3. Bread\n")
    f.write("4. Coffee (the most important one)\n")

print("\nShopping list completed. Let's verify:")
with open("file_handling_demo/shopping_list.txt", "r", encoding="utf-8") as f:
    print(f.read())


Shopping list completed. Let's verify:
My Shopping List:
1. Milk
2. Eggs
3. Bread
4. Coffee (the most important one)



In [25]:
# writelines(): writing multiple lines at once
students = [
    "Alice's Score: 95\n",
    "Bob's Score: 87\n",
    "Carol's Score: 92\n",
    "Dave's Score: 78\n",
    "Eve's Score: 99\n",
]

with open("file_handling_demo/students.txt", "w", encoding="utf-8") as f:
    f.write("Student Scores:\n")
    f.writelines(students)  # Write all at once

print("Student file completed. Here are the contents:")
with open("file_handling_demo/students.txt", "r", encoding="utf-8") as f:
    print(f.read())

Student file completed. Here are the contents:
Student Scores:
Alice's Score: 95
Bob's Score: 87
Carol's Score: 92
Dave's Score: 78
Eve's Score: 99



In [27]:
# 'a' mode to Append: add to end without erasing existing content
print("Before appending, the orignial text:")
with open("file_handling_demo/shopping_list.txt", "r", encoding="utf-8") as f:
    print(f.read())

# Now add more items
with open("file_handling_demo/shopping_list.txt", "a", encoding="utf-8") as f:
    f.write("5. Chocolate\n")
    f.write("6. Pizza\n")

print("After appending:")
with open("file_handling_demo/shopping_list.txt", "r", encoding="utf-8") as f:
    print(f.read())

Before appending, the orignial text:
My Shopping List:
1. Milk
2. Eggs
3. Bread
4. Coffee (the most important one)

After appending:
My Shopping List:
1. Milk
2. Eggs
3. Bread
4. Coffee (the most important one)
5. Chocolate
6. Pizza



### **Real-World Scenario: Writing Application Logs Like Google**

Every major tech company maintains **log files** which is a records of everything that happens in their systems. Google processes billions of search requests daily, and every error, warning, or event gets written to log files.

Here's how a simplified logging system might work:

In [30]:
import datetime

def write_log(log_file, level, message):
    """Write a timestamped log entry, just like production logging systems."""
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"[{timestamp}] [{level:7s}] {message}\n"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(entry)

log_path = "file_handling_demo/app.log"

# Simulate a web server handling requests
write_log(log_path, "INFO",    "Server started on port 8080")
write_log(log_path, "INFO",    "User 'alice@gmail.com' logged in")
write_log(log_path, "INFO",    "Search query: 'python file handling'")
write_log(log_path, "WARNING", "Response time exceeded 500ms for /api/search")
write_log(log_path, "INFO",    "User 'bob@gmail.com' logged in")
write_log(log_path, "ERROR",   "Database connection timeout, retrying...")
write_log(log_path, "INFO",    "Database reconnected successfully")

print("Application log written! Contents:")
print()
with open(log_path, "r", encoding="utf-8") as f:
    print(f.read())

Application log written! Contents:

[2026-05-10 21:33:55] [INFO   ] Server started on port 8080
[2026-05-10 21:33:55] [INFO   ] User 'alice@gmail.com' logged in
[2026-05-10 21:33:55] [INFO   ] Search query: 'python file handling'
[2026-05-10 21:33:55] [WARNING] Response time exceeded 500ms for /api/search
[2026-05-10 21:33:55] [INFO   ] User 'bob@gmail.com' logged in
[2026-05-10 21:33:55] [ERROR  ] Database connection timeout, retrying...
[2026-05-10 21:33:55] [INFO   ] Database reconnected successfully



### **5. Reading from a File**

Python offers four ways to read file content, each suited to different needs:

| Method | Returns | Use When |
|--------|---------|----------|
| `read()` | Entire file as one string | File is small, you need everything |
| `read(n)` | Next `n` characters | Processing chunks |
| `readline()` | Next line as a string | Processing line by line with control |
| `readlines()` | All lines as a list | You need all lines as a list |
| `for line in file` | Lines one at a time | **Most common! Memory-efficient** |


#### **read() : reads the entire file as one big string**

In [32]:
with open("file_handling_demo/learn.txt", "r", encoding="utf-8") as f:
    content = f.read()
    print("Type:", type(content))
    print("\nFull content:")
    print(content)

Type: <class 'str'>

Full content:
Day 1: Started learning Python file handling!
Day 2: Understood the 'with' statement. Life changing!
Day 3: Read and wrote my first CSV file like a data scientist!



#### **read(n) : read exactly n characters**

In [34]:
with open("file_handling_demo/learn.txt", "r", encoding="utf-8") as f:
    first_10 = f.read(10)
    print(f"First 10 characters: {repr(first_10)}") #Wraps the string in quotes and makes invisible chars like \n, \t visible 
    
    next_10 = f.read(10)
    print(f"Next 10 characters:  {repr(next_10)}")
    
    # Each call continues from where the last one stopped
    print("(Notice how each read() picks up where the last one left off)")

First 10 characters: 'Day 1: Sta'
Next 10 characters:  'rted learn'
(Notice how each read() picks up where the last one left off)


#### **readline() : reads one line at a time (includes the \n character)**

In [36]:
with open("file_handling_demo/learn.txt", "r", encoding="utf-8") as f:
    line1 = f.readline()
    line2 = f.readline()
    line3 = f.readline()
    line4 = f.readline()  # Empty string = end of file
    
    print("Line 1:", repr(line1))
    print("Line 2:", repr(line2))
    print("Line 3:", repr(line3))
    print("Line 4 (EOF):", repr(line4))  # Returns '' at end of file

Line 1: 'Day 1: Started learning Python file handling!\n'
Line 2: "Day 2: Understood the 'with' statement. Life changing!\n"
Line 3: 'Day 3: Read and wrote my first CSV file like a data scientist!\n'
Line 4 (EOF): ''


#### **readlines() : reads all lines and returns them as a list**

In [45]:
with open("file_handling_demo/learn.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
    
    print("Type:", type(lines)) # It shows that f.readlines() returns list of lines
    print("Number of lines:", len(lines))
    print("\nAll lines:")
    for i, line in enumerate(lines, 1):
        print(f"  Line {i}: {repr(line)}")

# Useful for processing specific lines:
print("\nFirst entry:", lines[0].strip())
print("Last entry:", lines[-1].strip())

Type: <class 'list'>
Number of lines: 3

All lines:
  Line 1: 'Day 1: Started learning Python file handling!\n'
  Line 2: "Day 2: Understood the 'with' statement. Life changing!\n"
  Line 3: 'Day 3: Read and wrote my first CSV file like a data scientist!\n'

First entry: Day 1: Started learning Python file handling!
Last entry: Day 3: Read and wrote my first CSV file like a data scientist!


#### **for loop : The best way for most cases**

In [48]:
# Memory-efficient: reads one line at a time without loading everything
with open("file_handling_demo/learn.txt", "r", encoding="utf-8") as f:
    print("Reading file entries:")
    print()
    for line_num, line in enumerate(f, 1):
        line = line.strip()  # Remove the trailing \n
        if line:             # Skip empty lines
            print(f"  Entry {line_num}: {line}")

print("\nThe for loop is memory-efficient, perfect for large files!")
print("Imagine reading a 10 GB server log, you can't load it all at once!")

Reading file entries:

  Entry 1: Day 1: Started learning Python file handling!
  Entry 2: Day 2: Understood the 'with' statement. Life changing!
  Entry 3: Day 3: Read and wrote my first CSV file like a data scientist!

The for loop is memory-efficient, perfect for large files!
Imagine reading a 10 GB server log, you can't load it all at once!


### Real-World Scenario: Reading User Data Like Netflix

Netflix needs to analyze user watch history to power its recommendation engine. They process millions of lines of watch history data stored in files. Here is a simplified version of how that might work:


In [50]:
# First, let's create some sample Netflix-style watch history
watch_history = [
    "alice,Stranger Things,S01E01,45:30,2024-01-15\n",
    "alice,Stranger Things,S01E02,50:22,2024-01-15\n",
    "alice,The Crown,S01E01,58:10,2024-01-16\n",
    "bob,Breaking Bad,S01E01,47:00,2024-01-15\n",
    "bob,Breaking Bad,S01E02,48:30,2024-01-16\n",
    "carol,Money Heist,S01E01,42:15,2024-01-17\n",
    "alice,Breaking Bad,S01E01,47:00,2024-01-17\n",
]

with open("file_handling_demo/watch_history.txt", "w", encoding="utf-8") as f:
    f.writelines(watch_history)

# Analyze: what shows does each user watch?
user_shows = {}

with open("file_handling_demo/watch_history.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        user, show = parts[0], parts[1]
        if user not in user_shows:
            user_shows[user] = set()
        user_shows[user].add(show)

print("Netflix-style User Preferences:")
for user, shows in user_shows.items():
    print(f"  {user}: {', '.join(sorted(shows))}")

print("\nThis is the foundation of recommendation systems!")

Netflix-style User Preferences:
  alice: Breaking Bad, Stranger Things, The Crown
  bob: Breaking Bad
  carol: Money Heist

This is the foundation of recommendation systems!


### **6. Navigating Inside a File : `seek()` and `tell()`**

When you read from a file, Python tracks your position with an invisible **cursor**, just like the blinking cursor in a text editor. Every character you read moves the cursor forward.

- **`tell()`** : "Where is the cursor right now?" Returns the current byte position.
- **`seek(position)`** : "Move the cursor to this position." Like teleporting!

Think of a file like a cassette tape:
- `tell()` shows which second of the tape you're at
- `seek()` lets you fast-forward or rewind to any point

**`seek(offset, whence)` : detailed signature:**
| `whence` value | Meaning |
|---|---|
| `0` (default) | From the beginning of the file |
| `1` | From the current position |
| `2` | From the end of the file |


In [54]:
with open("file_handling_demo/learn.txt", "r", encoding="utf-8") as f:
    
    print("Starting position:", f.tell())     # 0 at the beginning
    
    first_line = f.readline()
    print("After reading line 1:", f.tell())   # Moved forward!
    print("Line 1 was:", repr(first_line))
    
    # Read another line
    second_line = f.readline()
    print("After reading line 2:", f.tell())
    
    # Go back to the very beginning
    f.seek(0)
    print("\nAfter seek(0):", f.tell())       # Back to 0!
    
    # Read again from the start
    line_again = f.readline()
    print("Line 1 again:", repr(line_again))
    
    # Jump to a specific position (e.g., skip the first 7 characters "Day 1: ")
    f.seek(7)
    print("\nAfter seek(7):", repr(f.read(8)))  # Reads 8 chars from position 7

Starting position: 0
After reading line 1: 46
Line 1 was: 'Day 1: Started learning Python file handling!\n'
After reading line 2: 101

After seek(0): 0
Line 1 again: 'Day 1: Started learning Python file handling!\n'

After seek(7): 'Started '


### **7. File Object Properties**

Once you open a file, the file object comes with useful **properties** that let you inspect its state:

| Property / Method | Description |
|---|---|
| `.name` | The name/path of the file |
| `.mode` | The mode the file was opened in |
| `.closed` | `True` if the file is closed, `False` if open |
| `.readable()` | `True` if the file can be read |
| `.writable()` | `True` if the file can be written to |
| `.seekable()` | `True` if `seek()` is supported |


In [57]:
with open("file_handling_demo/learn.txt", "r+", encoding="utf-8") as f:
    print("File Object Properties: ")
    print(f"  .name       : {f.name}")
    print(f"  .mode       : {f.mode}")
    print(f"  .closed     : {f.closed}")
    print(f"  .readable() : {f.readable()}")
    print(f"  .writable() : {f.writable()}")
    print(f"  .seekable() : {f.seekable()}")

# After the 'with' block, the file is closed
print(f"\nAfter 'with' block:")
print(f"  .closed     : {f.closed}")
print("The file was automatically closed!")

File Object Properties: 
  .name       : file_handling_demo/learn.txt
  .mode       : r+
  .closed     : False
  .readable() : True
  .writable() : True
  .seekable() : True

After 'with' block:
  .closed     : True
The file was automatically closed!


### **8. File & Directory Management : The `os` Module**

The `os` module is Python's interface to your operating system. It lets you:
- Navigate the file system (like writing `cd` in the terminal)
- Create and delete directories
- Rename and delete files
- List directory contents

Think of the `os` module as your **File Explorer / Finder**, but controlled by Python code.

### Key Functions at a Glance

| Function | What It Does |
|---|---|
| `os.getcwd()` | Get current working directory |
| `os.chdir(path)` | Change directory |
| `os.listdir(path)` | List files and folders |
| `os.mkdir(path)` | Create one directory |
| `os.makedirs(path)` | Create directory + all parent directories |
| `os.rmdir(path)` | Remove an **empty** directory |
| `os.rename(src, dst)` | Rename a file or directory |
| `os.remove(path)` | Delete a file |
| `os.path.exists(path)` | Check if file/folder exists |
| `os.path.isfile(path)` | Check if it's a file |
| `os.path.isdir(path)` | Check if it's a directory |
| `os.path.join(...)` | Build paths safely (works on all OS) |
| `os.path.getsize(path)` | Get file size in bytes |
| `os.path.abspath(path)` | Get absolute path |
| `os.path.basename(path)` | Get filename from path |
| `os.path.dirname(path)` | Get directory from path |

In [66]:
import os

# Where are we right now?
current_dir = os.getcwd()
print("Current directory:", current_dir)

# What files and folders are in the demo folder?
print("\nContents of file_handling_demo/: ")
for item in os.listdir("file_handling_demo"):
    size = os.path.getsize(f"file_handling_demo/{item}")
    print(f"  {item:18s}  ({size} bytes)")

Current directory: /Users/kumarshikhar/ML Projects/OOPs

Contents of file_handling_demo/: 
  watch_history.txt   (299 bytes)
  shopping_list.txt   (100 bytes)
  students.txt        (101 bytes)
  temp_w.txt          (0 bytes)
  temp_a.txt          (0 bytes)
  app.log             (473 bytes)
  brand_new.txt       (23 bytes)
  learn.txt           (164 bytes)


In [ ]:
import os

# Create a directory
os.mkdir("file_handling_demo/reports")
print("Created: file_handling_demo/reports/")

# makedirs() creates all intermediate directories at once
os.makedirs("file_handling_demo/archive/2026/january", exist_ok=True)
print("Created: file_handling_demo/archive/2026/january/") #created 3 directories: archive, 2026 & january at once

# exist_ok=True tells os.makedirs() to silently do nothing if the directory already exists, instead of crashing with an error
os.makedirs("file_handling_demo/reports", exist_ok=True)

By default, **exist_ok is False**. So if the directory already exists, Python raises a **FileExistsError** and stops execution

In [75]:
import os

# Create a test file to play with
with open("file_handling_demo/old_name.txt", "w") as f:
    f.write("I will be renamed!\n")

print("Before rename:", os.path.exists("file_handling_demo/old_name.txt"))

# Rename the file
os.rename("file_handling_demo/old_name.txt", "file_handling_demo/new_name.txt")
print("After rename: ", os.path.exists("file_handling_demo/old_name.txt")) #old_name
print("After rename: ", os.path.exists("file_handling_demo/new_name.txt"))

# Safe deletion: always check before removing!
if os.path.exists("file_handling_demo/new_name.txt"):
    os.remove("file_handling_demo/new_name.txt")
    print("File removed safely!")

# Remove empty directories
os.rmdir("file_handling_demo/reports")
print("Empty directory removed!")

Before rename: True
After rename:  False
After rename:  True
File removed safely!
Empty directory removed!


In [81]:
import os

# os.path for working with file paths
file_path = "file_handling_demo/learn.txt"

print("os.path Functions: ")
print(f"  exists()   : {os.path.exists(file_path)}")
print(f"  isfile()   : {os.path.isfile(file_path)}")
print(f"  isdir()    : {os.path.isdir(file_path)}")
print(f"  getsize()  : {os.path.getsize(file_path)} bytes")
print(f"  abspath()  : {os.path.abspath(file_path)}")
print(f"  basename() : {os.path.basename(file_path)}")
print(f"  dirname()  : {os.path.dirname(file_path)}")

# os.path.join() build paths that work on all operating systems
# On Windows, separators are \, on Mac/Linux they're /
safe_path = os.path.join("file_handling_demo", "reports", "summary.txt")
print(f"\n os.path.join(): {safe_path}")

os.path Functions: 
  exists()   : True
  isfile()   : True
  isdir()    : False
  getsize()  : 164 bytes
  abspath()  : /Users/kumarshikhar/ML Projects/OOPs/file_handling_demo/learn.txt
  basename() : learn.txt
  dirname()  : file_handling_demo

 os.path.join(): file_handling_demo/reports/summary.txt


### **9. Modern Path Handling : `pathlib` (Python 3.4+)**

While `os.path` works, Python 3.4 introduced `pathlib` which is a much more elegant, object-oriented way to work with file paths.

Instead of writing:
```python
os.path.join("folder", "subfolder", "file.txt")
```
You write:
```python
Path("folder") / "subfolder" / "file.txt"
```

The `/` operator for joining paths. `pathlib.Path` objects know they're paths. They have smart methods built in, making your code cleaner and more readable.

In [86]:
from pathlib import Path

# Create a Path object
p = Path("file_handling_demo/learn.txt")

print("pathlib.Path Properties:")
print(f"  Path object : {p}")
print(f"  .name       : {p.name}")        # filename with extension
print(f"  .stem       : {p.stem}")        # filename without extension
print(f"  .suffix     : {p.suffix}")      # just the extension
print(f"  .parent     : {p.parent}")      # parent directory
print(f"  .exists()   : {p.exists()}")
print(f"  .is_file()  : {p.is_file()}")
print(f"  .is_dir()   : {p.is_dir()}")
print(f"  .stat().st_size: {p.stat().st_size} bytes")

# Join paths with / operator
base = Path("file_handling_demo")
log_path = base / "logs" / "server.log"
print(f"\nJoined path: {log_path}")

# Glob is used to find files matching a pattern
print("\nAll .txt files in file_handling_demo/:")
for txt_file in Path("file_handling_demo").glob("*.txt"):
    print(f"  {txt_file.name}")

pathlib.Path Properties:
  Path object : file_handling_demo/learn.txt
  .name       : learn.txt
  .stem       : learn
  .suffix     : .txt
  .parent     : file_handling_demo
  .exists()   : True
  .is_file()  : True
  .is_dir()   : False
  .stat().st_size: 164 bytes

Joined path: file_handling_demo/logs/server.log

All .txt files in file_handling_demo/:
  watch_history.txt
  shopping_list.txt
  students.txt
  temp_w.txt
  temp_a.txt
  brand_new.txt
  learn.txt


In [88]:
from pathlib import Path

# pathlib can read and write files directly!
p = Path("file_handling_demo/pathlib_test.txt")

# Write text directly with p.write_text()
p.write_text("Hello from pathlib!\nNo open() needed!\n", encoding="utf-8")

# Read text directly with .read_text()
content = p.read_text(encoding="utf-8")
print(content)

# Create directories
new_dir = Path("file_handling_demo/pathlib_dir")
new_dir.mkdir(exist_ok=True)
print(f"Directory created: {new_dir}")

# Clean up
p.unlink()          # Delete the file
new_dir.rmdir()     # Remove the directory
print("Cleaned up!")

Hello from pathlib!
No open() needed!

Directory created: file_handling_demo/pathlib_dir
Cleaned up!


### **10. Working with CSV Files**

CSV (Comma-Separated Values) is the most universal data format in the world. Open any spreadsheet, export from any database, download from any API, the chances are it gives you CSV.

**Why not just use `open()` and split by commas?**
Because CSV has edge cases: values with commas inside them, quoted strings, escaped characters, different delimiters. The `csv` module handles all of this for you.

### The CSV Module's Four Tools

| Tool | Use Case |
|------|----------|
| `csv.reader` | Read CSV rows as lists |
| `csv.writer` | Write lists as CSV rows |
| `csv.DictReader` | Read CSV rows as dictionaries (column names as keys) |
| `csv.DictWriter` | Write dictionaries as CSV rows |

In [91]:
import csv

# csv.writer to write rows as lists
employees = [
    ["Name", "Department", "Salary", "Years"],
    ["Alice Johnson", "Engineering", 95000, 5],
    ["Bob Smith", "Marketing", 72000, 3],
    ["Carol White", "Engineering", 105000, 8],
    ["Dave Brown", "HR", 65000, 2],
    ["Eve Davis, PhD", "Research", 115000, 10],
]

with open("file_handling_demo/employees.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(employees)  # Write all rows at once

print("CSV file written. Let's see the raw content:")
with open("file_handling_demo/employees.csv", "r", encoding="utf-8") as f:
    print(f.read())

CSV file written. Let's see the raw content:
Name,Department,Salary,Years
Alice Johnson,Engineering,95000,5
Bob Smith,Marketing,72000,3
Carol White,Engineering,105000,8
Dave Brown,HR,65000,2
"Eve Davis, PhD",Research,115000,10



In [103]:
import csv

# csv.reader — read rows as lists
print("Reading with csv.reader:")
print()

with open("file_handling_demo/employees.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)   # Read the first row as header
    print(f"Columns: {header}")
    print()
    
    for row in reader:
        name, dept, salary, years = row
        print(f"  {name:<15} | {dept:<11} | ${int(salary):>7,} | {years} yrs")

Reading with csv.reader:

Columns: ['Name', 'Department', 'Salary', 'Years']

  Alice Johnson   | Engineering | $ 95,000 | 5 yrs
  Bob Smith       | Marketing   | $ 72,000 | 3 yrs
  Carol White     | Engineering | $105,000 | 8 yrs
  Dave Brown      | HR          | $ 65,000 | 2 yrs
  Eve Davis, PhD  | Research    | $115,000 | 10 yrs


In [105]:
import csv

# csv.DictReader reads each row as a dictionary!
# Column headers become the keys automatically
print("Reading with csv.DictReader:")
print()

total_salary = 0
engineers = []

with open("file_handling_demo/employees.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        # Access values by column name as it is much more readable!
        print(f"  {row['Name']}: {row['Department']} (${int(row['Salary']):,})") # :, adds comma-separator for thousands so 95000 -> 95,000
        total_salary += int(row["Salary"])
        if row["Department"] == "Engineering":
            engineers.append(row["Name"])

print(f"\nTotal salary bill: ${total_salary:,}")
print(f"Engineers: {', '.join(engineers)}")

Reading with csv.DictReader:

  Alice Johnson: Engineering ($95,000)
  Bob Smith: Marketing ($72,000)
  Carol White: Engineering ($105,000)
  Dave Brown: HR ($65,000)
  Eve Davis, PhD: Research ($115,000)

Total salary bill: $452,000
Engineers: Alice Johnson, Carol White


In [107]:
import csv

# csv.DictWriter to write dictionaries as CSV rows
new_employees = [
    {"Name": "Frank Lee", "Department": "Engineering", "Salary": 98000, "Years": 4},
    {"Name": "Grace Kim", "Department": "Design", "Salary": 82000, "Years": 6},
]

fieldnames = ["Name", "Department", "Salary", "Years"]

with open("file_handling_demo/new_employees.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()     # Writes the column header row
    writer.writerows(new_employees)

print("New employees CSV created!")
with open("file_handling_demo/new_employees.csv", "r", encoding="utf-8") as f:
    print(f.read())

New employees CSV created!
Name,Department,Salary,Years
Frank Lee,Engineering,98000,4
Grace Kim,Design,82000,6



### Real-World Scenario: Processing Orders Like Amazon

Amazon processes millions of orders. Each order record is often stored as CSV data. Here's a simplified order analytics pipeline:


In [109]:
import csv
from collections import defaultdict

# Create sample order data (like Amazon's order database export)
orders = [
    ["order_id", "customer", "product", "category", "price", "quantity"],
    ["A001", "Alice", "Python Book", "Books", 29.99, 2],
    ["A002", "Bob", "Wireless Mouse", "Electronics", 45.00, 1],
    ["A003", "Alice", "Coffee Maker", "Appliances", 89.99, 1],
    ["A004", "Carol", "Python Book", "Books", 29.99, 3],
    ["A005", "Bob", "Keyboard", "Electronics", 75.00, 1],
    ["A006", "Dave", "Coffee Maker", "Appliances", 89.99, 2],
    ["A007", "Alice", "Headphones", "Electronics", 120.00, 1],
]

with open("file_handling_demo/orders.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(orders)

# Analyze: revenue by category
revenue_by_category = defaultdict(float)
orders_by_customer = defaultdict(int)

with open("file_handling_demo/orders.csv", "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        revenue = float(row["price"]) * int(row["quantity"])
        revenue_by_category[row["category"]] += revenue
        orders_by_customer[row["customer"]] += 1

print("Amazon-style Sales Report: ")
print("\nRevenue by Category:")
for cat, rev in sorted(revenue_by_category.items(), key=lambda x: -x[1]):
    print(f"  {cat:<15}: ${rev:>8.2f}")

print("\nOrders by Customer:")
for customer, count in sorted(orders_by_customer.items()):
    print(f"  {customer:<10}: {count} orders")

Amazon-style Sales Report: 

Revenue by Category:
  Appliances     : $  269.97
  Electronics    : $  240.00
  Books          : $  149.95

Orders by Customer:
  Alice     : 3 orders
  Bob       : 2 orders
  Carol     : 1 orders
  Dave      : 1 orders


### 11. Working with JSON Files

JSON (JavaScript Object Notation) is the language of the internet. Every web API, every mobile app config, every modern web service communicates using JSON.

JSON maps naturally to Python:

| JSON | Python |
|------|--------|
| `object {}` | `dict` |
| `array []` | `list` |
| `string ""` | `str` |
| `number` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

### The Four JSON Functions

| Function | Direction | Use Case |
|----------|-----------|----------|
| `json.dump(obj, file)` | Python → File | Save Python data to a JSON file |
| `json.load(file)` | File → Python | Load JSON from a file into Python |
| `json.dumps(obj)` | Python → String | Convert Python to JSON string |
| `json.loads(string)` | String → Python | Parse JSON string into Python |

**Memory trick:** `dump`/**`load`** work with **files**. `dumps`/**`loads`** work with **strings** (the 's' stands for 'string'!)


In [112]:
import json

# Let's save data from a python dictionary to json
user_profile = {
    "username": "alice_wonder",
    "email": "alice@example.com",
    "age": 28,
    "premium": True,
    "watch_history": ["Stranger Things", "The Crown", "Breaking Bad"],
    "settings": {
        "autoplay": True,
        "quality": "4K",
        "language": "English"
    },
    "last_login": None
}

# json.dump() to save Python dict to a JSON file
with open("file_handling_demo/user_profile.json", "w", encoding="utf-8") as f:
    json.dump(user_profile, f, indent=4)   # indent=4 makes it easy to read

print("Saved user profile to JSON file!")

print("\nRaw file contents:")
with open("file_handling_demo/user_profile.json", "r", encoding="utf-8") as f:
    print(f.read())

Saved user profile to JSON file!

Raw file contents:
{
    "username": "alice_wonder",
    "email": "alice@example.com",
    "age": 28,
    "premium": true,
    "watch_history": [
        "Stranger Things",
        "The Crown",
        "Breaking Bad"
    ],
    "settings": {
        "autoplay": true,
        "quality": "4K",
        "language": "English"
    },
    "last_login": null
}


In [114]:
import json

# json.load() to load JSON file back into Python
with open("file_handling_demo/user_profile.json", "r", encoding="utf-8") as f:
    loaded_profile = json.load(f)

print("Loaded profile type:", type(loaded_profile))

print("\n Accessing data:")
print(f"  Username : {loaded_profile['username']}")
print(f"  Premium  : {loaded_profile['premium']}")
print(f"  Quality  : {loaded_profile['settings']['quality']}")
print(f"  Watched  : {', '.join(loaded_profile['watch_history'])}")

print("\n Notice: Python types are preserved perfectly!")
print(f"  'premium' is type: {type(loaded_profile['premium'])}")   # bool, not string
print(f"  'age' is type: {type(loaded_profile['age'])}")           # int, not string

Loaded profile type: <class 'dict'>

Accessing data:
  Username : alice_wonder
  Premium  : True
  Quality  : 4K
  Watched  : Stranger Things, The Crown, Breaking Bad

Notice: Python types are preserved perfectly!
  'premium' is type: <class 'bool'>
  'age' is type: <class 'int'>


In [116]:
import json

# json.dumps() — Python to JSON string (useful for APIs, printing, logging)
data = {"name": "Bob", "scores": [95, 87, 92], "passed": True}

json_string = json.dumps(data)
print("json.dumps() result:")
print(f"  Type  : {type(json_string)}")
print(f"  Value : {json_string}")

# Pretty-print with indentation
pretty = json.dumps(data, indent=2, sort_keys=True)
print("\nPretty version:")
print(pretty)

# json.loads() : JSON string back to Python (useful when receiving API responses)
api_response = '{"status": "ok", "results": 42, "data": [1, 2, 3]}'
parsed = json.loads(api_response)

print("\njson.loads() result:")
print(f"  Type    : {type(parsed)}")
print(f"  Status  : {parsed['status']}")
print(f"  Results : {parsed['results']}")


json.dumps() result:
  Type  : <class 'str'>
  Value : {"name": "Bob", "scores": [95, 87, 92], "passed": true}

Pretty version:
{
  "name": "Bob",
  "passed": true,
  "scores": [
    95,
    87,
    92
  ]
}

json.loads() result:
  Type    : <class 'dict'>
  Status  : ok
  Results : 42


### Real-World Scenario: App Settings Like YouTube/Spotify

Every app stores user settings and configuration in JSON files. Here's how a music app might save and load user preferences:


In [ ]:
import json
import os

SETTINGS_FILE = "file_handling_demo/spotify_settings.json"

DEFAULT_SETTINGS = {
    "audio": {
        "quality": "high",
        "normalize": True,
        "equalizer": "flat"
    },
    "ui": {
        "theme": "dark",
        "language": "en",
        "show_lyrics": True
    },
    "social": {
        "share_activity": False,
        "follow_artists": True
    }
}

def load_settings():
    """Load settings from file, falling back to defaults if file doesn't exist."""
    if os.path.exists(SETTINGS_FILE):
        with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return DEFAULT_SETTINGS.copy()

def save_settings(settings):
    """Save settings to file."""
    with open(SETTINGS_FILE, "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=4)
    print("Settings saved!")

def update_setting(settings, section, key, value):
    """Update a specific setting."""
    settings[section][key] = value
    return settings

# Load initial settings
settings = load_settings()
print("Loaded settings:")
print(f"  Theme   : {settings['ui']['theme']}")
print(f"  Quality : {settings['audio']['quality']}")

# User changes a setting
settings = update_setting(settings, "ui", "theme", "light")
settings = update_setting(settings, "audio", "quality", "ultra")

# Save updated settings
save_settings(settings)

# Verify it persisted
reloaded = load_settings()
print("\nAfter reload:")
print(f"  Theme   : {reloaded['ui']['theme']}")
print(f"  Quality : {reloaded['audio']['quality']}")
print("\nSettings persist across app restarts — just like Spotify!")

### **12. Handling File Errors Gracefully**

File operations can fail for many reasons. A good program handles these failures gracefully instead of crashing.

### Common File Exceptions

| Exception | When It Occurs |
|-----------|---------------|
| `FileNotFoundError` | File or directory does not exist |
| `PermissionError` | No permission to read/write the file |
| `FileExistsError` | File already exists (with `'x'` mode) |
| `IsADirectoryError` | Expected a file but found a directory |
| `IOError` / `OSError` | General I/O errors |
| `UnicodeDecodeError` | File encoding doesn't match the specified encoding |

**The golden rule:** Only catch exceptions you can meaningfully handle. Don't silently swallow errors!

In [122]:
# FileNotFoundError : the most common file error
print("FileNotFoundError: ")
try:
    with open("file_handling_demo/ghost_file.txt", "r") as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"  Caught: {e}")

# FileExistsError while creating a file that already exists
print("\n FileExistsError: ")
try:
    with open("file_handling_demo/learn.txt", "x") as f:
        f.write("this will fail")
except FileExistsError as e:
    print(f"  Caught: {e}")

# UnicodeDecodeError
print("\n UnicodeDecodeError: ")
# Write a file with latin-1 encoding, then try to read as ascii
with open("file_handling_demo/encoding_test.txt", "w", encoding="latin-1") as f:
    f.write("Café and naïve")
try:
    with open("file_handling_demo/encoding_test.txt", "r", encoding="ascii") as f:
        content = f.read()
except UnicodeDecodeError as e:
    print(f"  Caught: {type(e).__name__}")
    print("  What to do: specify the correct encoding (try utf-8 or latin-1)")
    # The fix:
    with open("file_handling_demo/encoding_test.txt", "r", encoding="latin-1") as f:
        print(f"  Fixed the error, Content: {f.read()}")

FileNotFoundError: 
  Caught: [Errno 2] No such file or directory: 'file_handling_demo/ghost_file.txt'

 FileExistsError: 
  Caught: [Errno 17] File exists: 'file_handling_demo/learn.txt'

 UnicodeDecodeError: 
  Caught: UnicodeDecodeError
  What to do: specify the correct encoding (try utf-8 or latin-1)
  Fixed the error, Content: Café and naïve


## **Summary : Your Python File Handling Cheat Sheet**

### Opening Files
```python
# Always use 'with' as it auto-closes the file
with open("file.txt", "r", encoding="utf-8") as f:
    ...
```

### Reading
```python
f.read()          # Entire file as a string
f.read(n)         # Next n characters
f.readline()      # One line at a time
f.readlines()     # All lines as a list
for line in f:    # Best for large files and memory efficient
    ...
```

### Writing
```python
f.write("text")          # Write a string
f.writelines(list)       # Write a list of strings
# Remember: write() does NOT add newlines automatically!
```

### Navigation
```python
f.tell()           # Current cursor position (bytes)
f.seek(0)          # Jump to beginning
f.seek(0, 2)       # Jump to end
```

### OS Module
```python
os.getcwd()                    # Current directory
os.listdir("path")             # List directory contents
os.mkdir("path")               # Create directory
os.makedirs("a/b/c", exist_ok=True)  # Create nested directories
os.rename("old", "new")        # Rename file/directory
os.remove("file.txt")          # Delete a file
os.path.exists("path")         # Check if path exists
os.path.join("a", "b", "c")   # Build safe cross-platform paths
os.path.getsize("file")        # File size in bytes
```

### CSV
```python
import csv
# Reading
with open("data.csv") as f:
    for row in csv.reader(f): ...          # rows as lists
    for row in csv.DictReader(f): ...      # rows as dicts

# Writing
with open("data.csv", "w", newline="") as f:
    csv.writer(f).writerows(data)          # from lists
    w = csv.DictWriter(f, fieldnames=[...])
    w.writeheader(); w.writerows(data)     # from dicts
```

### JSON
```python
import json
json.dump(data, file, indent=4)    # Python → JSON file
json.load(file)                    # JSON file → Python
json.dumps(data)                   # Python → JSON string
json.loads(string)                 # JSON string → Python
```

### pathlib (Modern Path Handling)
```python
from pathlib import Path
p = Path("folder") / "subfolder" / "file.txt"
p.read_text(encoding="utf-8")
p.write_text("content", encoding="utf-8")
p.exists(), p.is_file(), p.is_dir()
list(p.parent.glob("*.txt"))
```

**Remember:** Files are how your programs talk to the world. Master them and you unlock data persistence, automation, data science, and backend development!